# Classical Solver for TSP and its Variants using Google's CP-SAT.

In [10]:
from ortools.sat.python import cp_model
import numpy as np

In [17]:
def solve_tsp_path_cpsat(distance_matrix, start=0, end=None):
    """
    Solves the shortest Hamiltonian path.
    - If start and end are both specified: fixed start and end.
    - If only start is specified: fixed start, free end.
    - If neither: completely free start and end.
    """
    from ortools.sat.python import cp_model

    n = len(distance_matrix)
    model  = cp_model.CpModel()
    solver = cp_model.CpSolver()

    # --- Arc variables (same as before) ---
    x = {}
    arcs = []
    for i in range(n):
        for j in range(n):
            if i != j:
                b = model.NewBoolVar(f"x_{i}_{j}")
                x[(i, j)] = b
                arcs.append((i, j, b))

    # --- Dummy return arcs: end → start, zero cost, to close the circuit ---
    # AddCircuit requires a circuit, so we add a free "ghost" return arc
    # for each possible (end_node → start_node) pair, but only allow
    # the correct one to be selected based on start/end parameters.

    dummy_arcs = {}
    for i in range(n):       # possible path end nodes
        for j in range(n):   # possible path start nodes
            if i != j:
                # Only create the ghost arc if it's consistent with
                # the start/end constraints
                skip = False
                if end is not None and i != end:
                    skip = True
                if start is not None and j != start:
                    skip = True
                if not skip:
                    b = model.NewBoolVar(f"dummy_{i}_{j}")
                    dummy_arcs[(i, j)] = b
                    arcs.append((i, j, b))  # zero-cost arc added to circuit

    # Exactly one dummy arc must be selected (closes the open path into a circuit)
    model.AddExactlyOne(dummy_arcs.values())

    # --- Circuit constraint over real + dummy arcs ---
    model.AddCircuit(arcs)

    # --- Objective: minimize real arc costs only (dummy arcs have zero cost) ---
    model.Minimize(
        sum(distance_matrix[i][j] * x[(i, j)]
            for i in range(n)
            for j in range(n) if i != j)
    )

    # --- Solve ---
    solver.parameters.log_search_progress = True
    status = solver.Solve(model)

    # --- Extract solution ---
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        # Find which dummy arc was selected → tells us actual start and end
        path_end   = None
        path_start = None
        for (i, j), var in dummy_arcs.items():
            if solver.Value(var) == 1:
                path_end   = i  # dummy goes FROM the path end
                path_start = j  # dummy goes TO the path start
                break

        # Reconstruct path by following real arcs from path_start
        next_node = {}
        for (i, j), var in x.items():
            if solver.Value(var) == 1:
                next_node[i] = j

        path = [path_start]
        while path[-1] != path_end:
            path.append(next_node[path[-1]])

        print(f"Status : {solver.StatusName(status)}")
        print(f"Length : {solver.ObjectiveValue()}")
        print(f"Path   : {path}")
        print(f"Start  : {path_start}, End: {path_end}")
        return path

    print("No solution found.")
    return None

In [18]:
W = [
    [  0.00,  26.57,  12.07,  47.19, 126.81,  35.69,  77.42,  76.09, 125.47,  78.93],
    [ 40.15,   0.00,  41.21,  78.54, 118.78,  41.20,  91.83,  85.67, 134.36, 140.95],
    [ 12.07,  70.52,   0.00,  39.15,  64.20,  92.26, 115.77, 117.75, 141.45,  39.78],
    [ 62.83,  40.30,  39.15,   0.00,  48.47,  15.35,  96.51,  49.51,  90.94,   0.77],
    [172.35, 134.38,  64.20,  27.94,   0.00,  48.81,  26.91,  23.45,  31.52,  28.75],
    [ 90.54,  41.20,  42.21,   3.04,  49.72,   0.00,  68.40,  48.17, 117.34,   8.62],
    [ 76.64,  89.36,  87.14,  52.94,  26.91,  94.27,   0.00,  12.55,   8.96,  67.32],
    [ 77.27,  85.52,  87.38,  52.82,  23.46,  48.17,   6.24,   0.00,   9.16,  67.09],
    [130.99, 133.05, 110.34,  73.83,  31.52,  83.61,  11.19,   8.96,   0.00,  58.48],
    [113.36,  74.74,  40.17,   0.77,  38.26,   8.56, 134.50,  65.68,  58.58,   0.00],
]

test = W[:4][:4]

solve_tsp_path_cpsat(test)


Starting CP-SAT solver v9.15.6755
Parameters: log_search_progress: true
Setting number of workers to 8

Initial optimization model '': (model_fingerprint: 0x41f2b4dfcf72139b)
#Variables: 15 (#bools: 12 in floating point objective) (14 primary variables)
  - 15 Booleans in [0,1]
#kCircuit: 1
#kExactlyOne: 1 (#literals: 3)

Starting presolve at 0.00s
[Scaling] Floating point objective has 12 terms with magnitude in [12.07, 78.54] average = 42.4792
[Scaling] Objective coefficient relative error: 2.22045e-16
[Scaling] Objective worst-case absolute error: 9.09495e-15
[Scaling] Objective scaling factor: 100
  1.10e-05s  0.00e+00d  [DetectDominanceRelations] 
  6.94e-04s  2.00e-08d  [PresolveToFixPoint] #num_loops=2 #num_dual_strengthening=1 
  3.00e-06s  0.00e+00d  [ExtractEncodingFromLinear] #potential_supersets=1 
  4.00e-06s  0.00e+00d  [DetectDuplicateColumns] 
  2.20e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 35 nodes and 48 arcs.
[Symmetry] Symmet

[0, 2, 3, 1]